# PCA Explained Variance Analysis — Per-Patch Morphable Models

Analyses how many PCA components are actually needed across all 32 superpoint models.
Variance per component is derived from `gt_z` (the pre-projected coefficients stored in `pca_basis_all.pth`).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D

SUPERPOINT_INDICES = [
    75, 411, 2699, 911, 8594, 3380, 6731, 9710, 9633, 119,
    3441, 6319, 9541, 8732, 6162, 3774, 8296, 3151, 10,
    7720, 6858, 7409, 7531, 3504, 6937, 4189, 8891, 3721,
    9241, 2213, 1765, 7547
]
THRESHOLDS = [0.80, 0.90, 0.95, 0.99]

data = torch.load('pca_basis_all.pth')
gt_z = data['gt_z'].numpy()   # [32, 8000, 100]
n_superpoints, n_samples, n_components = gt_z.shape
print(f'Superpoints: {n_superpoints}, Samples: {n_samples}, Components: {n_components}')

In [ ]:
# Variance of each PCA coefficient across all training samples
# PCA guarantees these are in decreasing order.
variances = np.var(gt_z, axis=1, ddof=1)   # [32, 100]
total_var = variances.sum(axis=1, keepdims=True)  # [32, 1]

# Cumulative explained variance ratio (relative to the 100-component subspace)
cum_evr = np.cumsum(variances / total_var, axis=1)  # [32, 100]

components = np.arange(1, n_components + 1)
print(f'Mean EVR at component 10: {cum_evr[:, 9].mean():.3f}')
print(f'Mean EVR at component 50: {cum_evr[:, 49].mean():.3f}')
print(f'Mean EVR at component 100: {cum_evr[:, 99].mean():.3f}')

## 1. Cumulative Explained Variance — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = cm.tab20(np.linspace(0, 1, n_superpoints))
for i in range(n_superpoints):
    ax.plot(components, cum_evr[i], color=colors[i], alpha=0.55, lw=1.2,
            label=f'SP {SUPERPOINT_INDICES[i]}')

mean_curve = cum_evr.mean(axis=0)
std_curve  = cum_evr.std(axis=0)
ax.plot(components, mean_curve, color='black', lw=2.5, label='Mean')
ax.fill_between(components, mean_curve - std_curve, mean_curve + std_curve,
                color='black', alpha=0.12, label='±1 std')

for thr in THRESHOLDS:
    ax.axhline(thr, ls='--', lw=0.9, color='gray')
    ax.text(n_components + 0.5, thr, f'{int(thr*100)}%', va='center', fontsize=8)

ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Cumulative Explained Variance Ratio')
ax.set_title('Cumulative EVR per Patch Model')
ax.set_xlim(1, n_components)
ax.set_ylim(0, 1.02)
ax.legend(loc='lower right', fontsize=6, ncol=4)
plt.tight_layout()
plt.savefig('pca_cumevr_all.png', dpi=150)
plt.show()

## 2. Components Needed per Threshold

In [ ]:
# For each superpoint and threshold, find the minimum number of components
def components_for_threshold(cum_evr_matrix, threshold):
    """Returns array of shape [n_superpoints] with min components to reach threshold."""
    exceeded = cum_evr_matrix >= threshold   # [32, 100] bool
    # argmax returns first True index; if never reached → n_components
    first_idx = np.where(exceeded.any(axis=1),
                         exceeded.argmax(axis=1) + 1,
                         n_components)
    return first_idx

needed = {thr: components_for_threshold(cum_evr, thr) for thr in THRESHOLDS}

print(f'{"Threshold":>10}  {"Min":>5}  {"Median":>7}  {"Mean":>7}  {"Max":>5}  {"All≤50":>8}')
for thr, arr in needed.items():
    print(f'{thr:>10.0%}  {arr.min():>5}  {np.median(arr):>7.1f}  {arr.mean():>7.1f}  {arr.max():>5}  {(arr <= 50).all()!s:>8}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: box plot of component counts per threshold ---
ax = axes[0]
box_data = [needed[thr] for thr in THRESHOLDS]
bp = ax.boxplot(box_data, patch_artist=True, notch=False)
palette = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticks(range(1, len(THRESHOLDS) + 1))
ax.set_xticklabels([f'{int(t*100)}%' for t in THRESHOLDS])
ax.set_xlabel('Explained Variance Threshold')
ax.set_ylabel('Components Required')
ax.set_title('Components Needed per Threshold (all 32 models)')
ax.yaxis.grid(True, linestyle='--', alpha=0.5)

# --- Right: scatter per superpoint for each threshold ---
ax2 = axes[1]
sp_ids = np.arange(n_superpoints)
for idx, (thr, color) in enumerate(zip(THRESHOLDS, palette)):
    ax2.scatter(sp_ids, needed[thr], label=f'{int(thr*100)}%', color=color,
                s=30, alpha=0.85, zorder=3)
ax2.set_xlabel('Superpoint index (by list order)')
ax2.set_ylabel('Components Required')
ax2.set_title('Per-Superpoint Component Requirement')
ax2.legend(title='Threshold')
ax2.yaxis.grid(True, linestyle='--', alpha=0.5)
ax2.set_xticks(sp_ids)
ax2.set_xticklabels([str(s) for s in SUPERPOINT_INDICES], rotation=90, fontsize=6)

plt.tight_layout()
plt.savefig('pca_components_needed.png', dpi=150)
plt.show()

## 3. Per-Component Variance Distribution (Scree-like)

In [ ]:
evr = variances / total_var   # individual (not cumulative), [32, 100]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Mean individual EVR (scree plot)
ax = axes[0]
mean_evr = evr.mean(axis=0)
std_evr  = evr.std(axis=0)
ax.bar(components, mean_evr, color='steelblue', alpha=0.8)
ax.fill_between(components, mean_evr - std_evr, np.clip(mean_evr + std_evr, 0, 1),
                color='steelblue', alpha=0.3)
ax.set_xlabel('Component')
ax.set_ylabel('Mean Individual EVR')
ax.set_title('Scree Plot (averaged across 32 models)')

# Heatmap: EVR per superpoint × component
ax2 = axes[1]
im = ax2.imshow(evr, aspect='auto', cmap='viridis',
                extent=[0.5, n_components + 0.5, n_superpoints - 0.5, -0.5])
plt.colorbar(im, ax=ax2, label='Individual EVR')
ax2.set_xlabel('Component')
ax2.set_ylabel('Superpoint (list index)')
ax2.set_title('EVR Heatmap: Superpoint × Component')

plt.tight_layout()
plt.savefig('pca_scree_heatmap.png', dpi=150)
plt.show()

## 4. Summary & Recommendation

In [ ]:
print('=' * 55)
print('  PCA Component Recommendation Summary')
print('=' * 55)
for thr in THRESHOLDS:
    arr = needed[thr]
    worst = arr.max()
    median = int(np.median(arr))
    print(f'  {int(thr*100)}% variance → median {median:3d} comps, '
          f'worst-case {worst:3d} comps (SP {SUPERPOINT_INDICES[arr.argmax()]})')

print()
# How many components to cover ALL models at 95%?
safe_95 = needed[0.95].max()
safe_90 = needed[0.90].max()
print(f'  To cover ALL 32 models at 90%: {safe_90} components')
print(f'  To cover ALL 32 models at 95%: {safe_95} components')
print()

# How much variance does N=50 capture on average?
for n in [10, 20, 30, 50]:
    mean_cov = cum_evr[:, n - 1].mean()
    min_cov  = cum_evr[:, n - 1].min()
    print(f'  {n:3d} components → mean EVR {mean_cov:.3f}, worst-case {min_cov:.3f}')

print('=' * 55)

In [ ]:
+l
